In [4]:
import cv2
import numpy as np

# Cargar imagen en escala de grises
imagen = cv2.imread("img/IMG_20191209_100620.jpg")
imagen_gray = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

# Suavizado para reducir ruido
imagen_gray = cv2.GaussianBlur(imagen_gray, (5, 5), 0)

# Binarización con Otsu: dados claros sobre fondo oscuro -> dados en blanco
_, imagen_bin = cv2.threshold(imagen_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Si los dados aparecen negros, invertir (depende de la imagen)
# En este caso asumimos que Otsu ya deja los dados en blanco

# Operación morfológica: apertura para eliminar pequeños puntos de ruido
kernel = np.ones((3,3), np.uint8)
imagen_bin = cv2.morphologyEx(imagen_bin, cv2.MORPH_OPEN, kernel)

# Etiquetar componentes (dados)
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(imagen_bin, 8, cv2.CV_32S)

# Parámetros para filtrar dados (área mínima)
area_min_dado = 1000

# Copia de la imagen original para dibujar
imagen_resultado = imagen.copy()

# Contador de dados
contador_dados = 0

# Recorrer cada componente (omitir el fondo: etiqueta 0)
for i in range(1, num_labels):
    x = stats[i, cv2.CC_STAT_LEFT]
    y = stats[i, cv2.CC_STAT_TOP]
    w = stats[i, cv2.CC_STAT_WIDTH]
    h = stats[i, cv2.CC_STAT_HEIGHT]
    area = stats[i, cv2.CC_STAT_AREA]
    cX, cY = centroids[i]

    if area < area_min_dado:
        continue   # es ruido, no un dado

    contador_dados += 1

    # Crear máscara para este dado (píxeles con la etiqueta i)
    mascara_dado = (labels == i).astype(np.uint8) * 255

    # Extraer ROI de la imagen en grises y de la máscara
    roi_gray = imagen_gray[y:y+h, x:x+w]
    roi_mask = mascara_dado[y:y+h, x:x+w]

    # En la ROI, poner los píxeles fuera del dado a blanco (255)
    roi_masked = roi_gray.copy()
    roi_masked[roi_mask == 0] = 255   # fondo blanco para no interferir en umbral

    # Detectar puntos negros (pips) dentro del dado
    # Usamos umbral de Otsu con inversión para que los puntos aparezcan blancos
    _, puntos_bin = cv2.threshold(roi_masked, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Opcional: pequeña apertura para separar puntos pegados (si es necesario)
    puntos_bin = cv2.morphologyEx(puntos_bin, cv2.MORPH_OPEN, np.ones((2,2), np.uint8))

    # Etiquetar los puntos
    num_puntos, labels_puntos, stats_puntos, centroids_puntos = cv2.connectedComponentsWithStats(puntos_bin, 8, cv2.CV_32S)

    # Filtrar puntos por área mínima (ajustar según el tamaño esperado)
    area_min_punto = 10
    puntos_validos = 0
    for j in range(1, num_puntos):   # omitir fondo
        area_punto = stats_puntos[j, cv2.CC_STAT_AREA]
        if area_punto > area_min_punto:
            puntos_validos += 1

    # Dibujar el dado y el número de puntos
    cv2.rectangle(imagen_resultado, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.circle(imagen_resultado, (int(cX), int(cY)), 4, (0, 0, 255), -1)
    cv2.putText(imagen_resultado, str(puntos_validos), (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

print(f"Total de dados detectados: {contador_dados}")

# Mostrar resultado
cv2.imshow('Dados con puntos', imagen_resultado)
cv2.waitKey(0)
cv2.destroyAllWindows()

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'
